# Incorporating Persistence in ML Pipelines
#### Author - Siddharth Setlur
In this tutorial, we're going to incorporate persistence features into a simple, interpretable ML model that classifies 3D shapes. This notebook is based on the [giotto-tda tutorial](https://github.com/giotto-ai/giotto-tda/blob/master/examples/classifying_shapes.ipynb).

We will work with a simple model called a Random Forest Classifier -- itself an extension of a basic Decision Tree classifier architecture. Although this simple model causes us to lose some accuracy compared to state-of-the-art (SOTA) deep learning architectures like the DNN's or CNN's, we gain both interpretability and speed. The entire training process may take under a minute on (say) a laptop CPU with 8GB RAM, *including* the computation of peristence.

Please complete the `## FINISH_ME ##` sections of code in the notebook; then run the code, saving the output.
**The first two problems (3 points) are due for hand-in as the first part of Homework 3, by 9:30am on Friday 20 March.**

| <p align='left'> Problem  (by Section number)                | <p align='left'> Marks possible |  <p align='left'> Marks awarded |
| ------------------------------------- | --- | --- |
| <p align='left'> 1. Geometric 3d shapes: import and examine  | <p align='left'> 1 | |
| <p align='left'> 2. Examining the data via persistence  | <p align='left'> 2 | |
| <p align='left'> 3. Vectorization of the peristence diagrams | <p align='left'> -- | -- |
| <p align='left'> 4. Creating and training a Random Forest Classifier | <p align='left'> -- | -- |
| <p align='left'> 5. Loading and examining a more complicated dataset  | <p align='left'> -- | -- |
| <p align='left'> 6. Training another Random Forest Classifier  | <p align='left'> -- | -- |
| <p align='left'> **Total** | <p align='left'> max **3** | |

Same as last week, make sure that you've included the folder ```helper_functions``` with the file ```generate_datasets.py``` in your working directory.

We will use scikit-learn to create the Random Forest Classifier. (We've already got ```sklearn``` in our ```agq-env``` environment.)
We also need package that computes peristence while allowing us to compute and keep track of gradients -- so that we can train our classifier even after TDA 'layers' have been included in it. For this person we 

We will use the python TDA package **giotto** instead of **gudhi**, because it has better incorporation with scikit-learn. The package name, which you need to intall it, is ```giotto-tda```; but it's imported (further below in the notebook) as ```gtda```.

You know what to do... from a terminal, enter your agq-env conda environment and run
```conda install -c conda-forge giotto-tda```  or (if the former doesn't work)  ```pip install giotto-tda```
Alternatively, open this Jupyter notebook from the agq-env environment, and run:

In [ ]:
!pip install giotto-tda

## 1. Geometric 3d shapes: import and examine

Let's make 30 point clouds by random sampling from some pre-determined geometric shapes:

In [ ]:
from helper_functions.generate_datasets import make_point_clouds
import numpy as np
#get the point clouds and their labels
point_clouds_basic, labels_basic = make_point_clouds(n_samples_per_shape=10, n_points=20, noise=0.5)


The first step is always to examine the dataset we have. Pethaps, the first thing to do is to find the shape of the point clouds and the labels. 

In [ ]:
point_clouds_basic.shape, labels_basic.shape

There are 30 labels, corresponding to the 30 shapes we have. Each shape is a (400,3) array. But how many distinct labels are there, i.e. how many distinct kinds of shapes are we working with?

In [ ]:
np.unique(labels_basic)

Let's plot a sample of each class

In [ ]:
from matplotlib import pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

In [ ]:
samples = []
samples_labels = []
#get a single sample for each label
for i in range(len(labels_basic)):
    if labels_basic[i] not in samples_labels:
        samples.append(## FINISH_ME ## )
        samples_labels.append(## FINISH_ME ## )
#Plot the point clouds on a 3D projection
fig = plt.figure()
# Create a figure with 3 subplots (one for each shape)
fig = plt.figure(figsize=(15, 5))
for i in range(## FINISH_ME ## ):
    ax = fig.add_subplot(1, 3, i+1, projection='3d')
    ax.scatter(## FINISH_ME ## ) #Hint - be careful  - you're plotting a 3D array!!
    ax.set_title(f'Shape {int(samples_labels[i])}')
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')

plt.tight_layout()
plt.show()


## 2. Examining the data via persistence

In [ ]:
from gtda.homology import VietorisRipsPersistence
from gtda.plotting import plot_diagram
homology_dimensions = [0, 1, 2]
#Giotto has a very handy function to compute the persistence diagrams. Given a point cloud, it computes the VR complexes and then the persistence diagrams in the dimensions specified. 
VR_PD = VietorisRipsPersistence(homology_dimensions=homology_dimensions, collapse_edges=True) #this is a class that computes the diagrams given a point cloud; here we intiialize it to compute persistence in the 0,1,2 dimensions
#compute the persistence diagrams for the point clouds
#fit the persistence diagram
pd0 = VR_PD.fit_transform(samples[0][None,:,:]) # shape 0
pd1 = ## FINISH_ME ##  # shape 1
pd2 = ## FINISH_ME ##  # shape 2

The VR class also comes witha nice plot function that plots the persistence diagrams

In [ ]:
VR_PD.plot(pd0) #diagram for shape 0

In [ ]:
VR_PD.plot(## FINISH_ME ## ) #diagram for shape 1

In [ ]:
VR_PD.plot(## FINISH_ME ## ) #diagram for shape 2

What are these shapes???  Use the persistence diagrams and/or visual inspection of the plots to answer... ```## FINISH ME ##```

## 3. Vectorization of the peristence diagrams

As we saw in the lecture, we need to compute "vectorizations" or "representations" of the persistence diagrams, in order to feed them into ML pipelines. Again Giotto makes our lives very easy by providing classes for a bunch of common representations. Here, we use the persistence landscape.

In [ ]:
from gtda.diagrams import PersistenceLandscape
landscape = PersistenceLandscape()

landscape_0 = landscape.fit_transform(pd0) #landscape for shape 0
landscape_1 = landscape.fit_transform(## FINISH_ME ## ) #landscape for shape 1
landscape_2 = ## FINISH_ME ##  #landscape for shape 2


There are also super nice plotting functions for visualization! Can you see why a classifier fed the data of the persistence landscapes would be able to very easily classify the shapes?

In [ ]:
landscape.plot(landscape_0) #landscape for shape 0

In [ ]:
landscape.plot(## FINISH_ME ## ) #landscape for shape 1

In [ ]:
landscape.plot(## FINISH_ME ## ) #landscape for shape 2

## 4. Creating and training a Random Forest Classifier

We now train a Random Forest Classifier using just the landscapes. 

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
#split the dataset into train and test sets
X_train, X_test, y_train, y_test = train_test_split(point_clouds_basic, labels_basic, test_size=0.2, random_state=42)
#Compute the persistence diagrams for the training and test sets
H_train = landscape.fit_transform(VR_PD.fit_transform(X_train))
H_test = ## FINISH_ME ## 

In [ ]:
CLF = RandomForestClassifier(n_estimators=100, random_state=0, oob_score=True)
#One issue is that we can't just feed 3 vectors in to the classifier, we can only feed scalars, so we sum along each of the landscapes,
# i.e. for each point cloud we have 3 landscapes which are 3 vectors each of legnth 100. We sum each of the vectors to get 3 numbers
# to feed into the classifier for each point cloud.
CLF.fit(H_train.sum(axis=2), y_train)
CLF.oob_score_

Just summing along the landscapes is very crude, but it works very well (actually it works perfectly), but we will soon see that this is not the case for real-world data and we'll have to get creative. Let's delve more into the statistics. 

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

# Import necessary libraries for evaluation metrics
import matplotlib.pyplot as plt

# Print out-of-bag score (accuracy)
print(f"Out-of-bag accuracy: {CLF.oob_score_:.4f}")

# Plot feature importance
plt.figure(figsize=(10, 6))
plt.bar(range(len(CLF.feature_importances_)), CLF.feature_importances_)
plt.xlabel('Feature Index')
plt.ylabel('Feature Importance')
plt.title('Random Forest Feature Importance')
plt.tight_layout()
plt.show()

# Get predictions
y_pred = CLF.predict(H_test.sum(axis=2))

# Plot confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Shape 0", "Shape 1", "Shape 2"])
plt.figure(figsize=(8, 6))
disp.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.show()

# Print classification report
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Shape 0", "Shape 1", "Shape 2"]))

The performance is amazing! How can we interpret the classifier (look at the feature importance plot)?

## 5. Loading and examining a more complicated dataset

Let's try a more complicated dataset. We use a 3D dataset from a [Princeton computer vision course](https://www.cs.princeton.edu/courses/archive/fall09/cos429/assignment3.html) comprised of 4 classes with 10 samples each i.e. 40 total clouds

In [ ]:
!conda install -c conda-forge openml -y

In [ ]:


from openml.datasets.functions import get_dataset
import pandas as pd
df = get_dataset('shapes').get_data(dataset_format='dataframe')[0]



Let's explore the dataframe

In [ ]:
df.head()

Looks like each row contains a point and the label telling us which point cloud it belongs to. Let's see what the labels are

In [ ]:
df['target'].unique()

Let's visualize each of the shapes

In [ ]:
human_sample = df.query('target == "human_arms_out0"')[["x", "y", "z"]].values
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.scatter(human_sample[:, 0], human_sample[:, 1], human_sample[:, 2])
ax.set_title('Human Point Cloud')
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
plt.show()

In [ ]:
vase_sample = df.query('target == "vase0"')[["x", "y", "z"]].values
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.scatter(## FINISH_ME ## 
ax.set_title('Vase Point Cloud')
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
plt.show()

In [ ]:
chair_sample = df.query('target == "dining_chair0"')[["x", "y", "z"]].values
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.scatter(## FINISH_ME ## )
ax.set_title('Chair Point Cloud')
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
plt.show()

In [ ]:
biplane_sample = df.query('target == "biplane0"')[["x", "y", "z"]].values
fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')
ax.scatter(## FINISH_ME ## )
ax.set_title('Biplane Point Cloud')
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
plt.show()

This is a weird way to label things - they've labelled each point cloud uniquely, i.e. we have human_arms_out0,...,human_arms_out9 and similarly for the other 3 classes. Somehow, we need to make a labelling array as we had in the toy example above, i.e. a 1-d numpy array of length 40 where each entry is either 0,1,2, or 3 depending on whether which class it belongs to

In [ ]:
labels = np.zeros(40) # array with 40 zeros
labels[10:20] = 1 # label the samples 10-20 as 1 corresponding to the vase
labels[20:30] = 2 # label the samples 20-30 as 2 corresponding to the chair
labels[30:] = 3 # label the samples 30-40 as 3 corresponding to the biplane

The weird labelling method does make it easier to extract a list of point clouds though! We can iterate over the unique labels of the df, since each df label corresponds to a unique point cloud 

In [ ]:
point_clouds = np.asarray(
    [
        df.query("target == @shape")[["x", "y", "z"]].values
        for shape in df["target"].unique()
    ]
)

## 6. Training another Random Forest Classifier

In [ ]:
homology_dimensions = [0, 1, 2]
VR_PD = VietorisRipsPersistence(homology_dimensions=homology_dimensions, collapse_edges=True)
landscape = PersistenceLandscape()
CLF = RandomForestClassifier(n_estimators=100, random_state=0, oob_score=True)
#fit the persistence diagram
#split the dataset into train and test sets
X_train, X_test, y_train, y_test = train_test_split(point_clouds, labels, test_size=0.2, random_state=42)
#fit the persistence diagram
H_train = ## FINISH_ME ## 
CLF.fit(## FINISH_ME ## , y_train) #Remember to sum! 
CLF.oob_score_

In [ ]:
H_test = landscape.fit_transform(VR_PD.fit_transform(X_test))
# Print out-of-bag score (accuracy)
print(f"Out-of-bag accuracy: {CLF.oob_score_:.4f}")

# Plot feature importance
plt.figure(figsize=(10, 6))
plt.bar(range(len(CLF.feature_importances_)), CLF.feature_importances_)
plt.xlabel('Feature Index')
plt.ylabel('Feature Importance')
plt.title('Random Forest Feature Importance')
plt.tight_layout()
plt.show()

# Get predictions
y_pred = CLF.predict(H_test.sum(axis=2))

# Plot confusion matrix
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Human", "Vase", "Chair", "Biplane"])
plt.figure(figsize=(8, 6))
disp.plot(cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.show()

# Print classification report
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=["Human", "Vase", "Chair", "Biplane"]))

You can add more features in order to improve performance, though our performance on such a small datatset using a simple classifier is already great! Look into changing/adding features using other vectorizations like PersistenceImage or Betti curves. See the [giotto documentation](https://giotto-ai.github.io/gtda-docs/latest/modules/diagrams.html#representations) for implementations of these features. (Hint - they work the same way as landscapes except fro a change in name.) You could also look into [features](https://giotto-ai.github.io/gtda-docs/latest/modules/diagrams.html#features) such as number of points in the diagram. Once you've decided on your feature, the pipeline is as follows 

(point_clouds, labels) -> (x_train, y_train) (x_test, y_test). 

Compute topological features - Landscape/Image(VR(x_train)) (or use the features such as number of points).

Do some thing like summing if you have multiple vectors as we did for the landscape, essentially you can feed as many scalars as you want into the classifier but not vectors. 

Train the classifier using clf.fit(train_features).

Compute topological features on the test set.

Predict using the classfier.

Display summary stats - Hint - you can basically copy the last cell displaying the statistics to do the last 2 steps with minor modifications depending on your pipeline.

Finally interpret the classifier and discuss why you think the feature you chose improced the performance. 